In [ ]:
import os
import json
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Same cache dir as training so we reuse the downloaded base model
CACHE_DIR = "/projects/prjs1308/huggingface/"

# Path to the fine-tuned adapter + tokenizer
ADAPTER_DIR = "/projects/prjs1308/africa_llm_data/results/inference_models/20260308_205449_simple_gemma3/simple_gemma3_4b_lr0.0001_seed42_20260308_205449"

# Load run-time config saved during fine-tuning
with open(os.path.join(ADAPTER_DIR, "run_config.json"), "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

base_model_id = run_cfg["model_id"]
system_prompt = run_cfg.get("system_prompt")
prompt_template = run_cfg["prompt_template"]
max_tokens = run_cfg.get("max_tokens", 4096)
default_max_new_tokens = run_cfg.get("max_new_tokens", 500)
targets_spec = run_cfg.get("targets_spec")

# Device and quantization config (match training as closely as possible)
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = (
    torch.bfloat16
    if device == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float16
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

# Load tokenizer/processor from adapter directory (use shared cache dir)
processor = AutoProcessor.from_pretrained(ADAPTER_DIR, cache_dir=CACHE_DIR)
tokenizer = getattr(processor, "tokenizer", processor)

# Load base Gemma-3 model and attach LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quant_config,
    device_map={"": 0} if device == "cuda" else None,
    torch_dtype=compute_dtype if device == "cuda" else None,
    cache_dir=CACHE_DIR,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
model.to(device)

print("Loaded base model:", base_model_id)
print("Using adapter from:", ADAPTER_DIR)
print("Device:", device)

In [ ]:
import sys, os

# Ensure project root is importable so we can use agent_utils
PROJECT_ROOT = "/home/fcool/africa_llm"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from agent_utils.gemma3_finetune_simple import _extract_pred_json


def build_chat_input(text: str) -> str:
    """Build chat-formatted prompt using the same template + system prompt as training."""
    instruction = prompt_template.format(text)
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": instruction})
    chat = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return chat


def generate_annotation(text: str, max_new_tokens: int | None = None):
    """Generate JSON annotation for a single post, returning (raw_text, parsed_json)."""
    if max_new_tokens is None:
        max_new_tokens = default_max_new_tokens

    chat = build_chat_input(text)
    inputs = tokenizer(chat, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Slice off the prompt tokens to get only the generated completion
    gen_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    completion = tokenizer.decode(gen_ids, skip_special_tokens=True)
    parsed = _extract_pred_json(completion)
    return completion, parsed


# Quick smoke test (replace with a real post)
example_text = "We are moving forward. We are not gonna slow down. We are fixing this country."
raw_out, parsed_out = generate_annotation(example_text)
print("RAW COMPLETION:\n", raw_out)
print("\nPARSED JSON:\n", parsed_out)

In [ ]:
import pandas as pd

# Path created by inference/organise_data.ipynb (relative to this notebook)
DATA_PATH = "data/inference_data.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows from {DATA_PATH}")

# Select the slice of rows to run inference on
start_idx = 0   # inclusive
end_idx = 10    # exclusive

subset = df.iloc[start_idx:end_idx].copy()
print(f"Running inference on rows [{start_idx}, {end_idx}) → {len(subset)} examples")

for idx, row in subset.iterrows():
    text = row["text"]
    raw_out, parsed_out = generate_annotation(text)

    print("\n" + "=" * 80)
    print(f"ROW {idx}")
    print("TEXT:")
    print(text)
    print("\nRAW COMPLETION:")
    print(raw_out)
    print("\nPARSED JSON:")
    print(parsed_out)